# Pregenant 代谢组学数据分析

本 Notebook 对孕妇代谢组学数据进行综合分析，包括：

1. **数据预处理** - 清洗代谢组学数据
2. **Benchmark 对比** - TabPFN 与其他 8 种机器学习方法的性能对比
3. **SHAP 解释性分析** - 特征重要性和代谢物贡献度分析
4. **可视化展示** - 结果图表生成

**数据集信息：**
- 样本数: 40
- 特征数: 127 (代谢物)
- 目标变量: Pregent (N/Y - 二分类)
- 特征类型: 代谢物浓度 (质谱数据)

In [ ]:
# 设置 matplotlib 后端和字体
import matplotlib
matplotlib.use('Agg')  # 非交互式后端，适合保存图片

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

# 设置图形样式
plt.style.use('default')
sns.set_palette("husl")

# 忽略警告
warnings.filterwarnings('ignore')

print("✅ 依赖库导入完成")
print(f"Matplotlib backend: {matplotlib.get_backend()}")

## 1. 数据加载与预处理

In [ ]:
# 加载数据
data_path = '../../dataset/preganent raw data 202603(1).csv'
df = pd.read_csv(data_path)

print(f"数据集形状: {df.shape}")
print(f"\n列名预览:")
print(df.columns.tolist()[:10], "...")
print(f"\n目标变量分布:")
print(df['Pregent'].value_counts())

In [ ]:
# 数据预处理
def preprocess_data(df):
    """
    预处理代谢组学数据
    - 移除不需要的列: name, Annotation, EM score
    - 处理缺失值 (N/A, NaN)
    - 转换目标变量为数值
    - 对数转换处理代谢物浓度的偏态分布
    """
    # 复制数据
    data = df.copy()
    
    # 移除不需要的列
    cols_to_drop = ['name', 'Annotation', 'EM score']
    data = data.drop(columns=[c for c in cols_to_drop if c in data.columns])
    
    # 处理目标变量
    data['Pregent'] = data['Pregent'].map({'N': 0, 'Y': 1})
    data = data.dropna(subset=['Pregent'])
    
    # 分离特征和目标
    y = data['Pregent'].astype(int)
    X = data.drop(columns=['Pregent'])
    
    # 处理特征中的缺失值和异常值
    X = X.replace('N/A', np.nan)
    X = X.replace('', np.nan)
    X = X.apply(pd.to_numeric, errors='coerce')
    
    # 用中位数填充缺失值
    X = X.fillna(X.median())
    
    # 对数转换 (处理代谢组学数据的偏态分布)
    # 添加小常数避免 log(0)
    X_log = np.log1p(X + 1e-10)
    
    return X_log, y, X.columns.tolist()

X, y, feature_names = preprocess_data(df)

print(f"\n预处理后数据:")
print(f"特征矩阵形状: {X.shape}")
print(f"目标分布: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"\n特征示例 (前5个): {feature_names[:5]}")

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape[0]} 样本")
print(f"测试集: {X_test.shape[0]} 样本")
print(f"\n训练集类别分布: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"测试集类别分布: {dict(zip(*np.unique(y_test, return_counts=True)))}")

## 2. Benchmark 对比 - 9 种分类方法

In [ ]:
# 导入所有模型
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

# 定义所有模型
models = {
    'TabPFN': TabPFNClassifier.create_default_for_version(
        ModelVersion.V2, n_estimators=4, device='cpu'
    ),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000)
}

print(f"加载了 {len(models)} 个模型")
print("模型列表:", list(models.keys()))

In [ ]:
# 评估函数
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """评估单个模型性能"""
    import time
    
    # 训练时间
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start
    
    # 预测
    start = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start
    
    # 预测概率 (用于 ROC-AUC)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    
    # 计算指标
    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else np.nan,
        'Fit_Time': fit_time,
        'Predict_Time': predict_time
    }
    
    return results, y_pred, y_proba

# 运行所有模型评估
results_list = []
predictions = {}
probabilities = {}

print("开始评估所有模型...\n" + "="*60)

for name, model in models.items():
    print(f"\n[{name}] ", end="")
    try:
        result, y_pred, y_proba = evaluate_model(
            name, model, X_train, X_test, y_train, y_test
        )
        results_list.append(result)
        predictions[name] = y_pred
        probabilities[name] = y_proba
        print(f"✓ Accuracy: {result['Accuracy']:.4f}, F1: {result['F1-Score']:.4f}")
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        continue

# 创建结果 DataFrame
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n" + "="*60)
print("评估完成!")

In [ ]:
# 显示结果表格
print("\n" + "="*80)
print("BENCHMARK 结果对比")
print("="*80)

display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Fit_Time']
results_display = results_df[display_cols].copy()

# 格式化显示
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    results_display[col] = results_display[col].apply(lambda x: f"{x:.4f}")
results_display['Fit_Time'] = results_display['Fit_Time'].apply(lambda x: f"{x:.3f}s")

print(results_display.to_string(index=False))

In [ ]:
# 可视化对比结果
print("\n生成 Benchmark 对比图...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors = ['#2196F3' if 'TabPFN' in m else '#90A4AE' for m in results_df['Model']]

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    
    # 按 metric 排序
    sorted_df = results_df.sort_values(metric, ascending=True)
    bar_colors = ['#2196F3' if 'TabPFN' in m else '#90A4AE' for m in sorted_df['Model']]
    
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=bar_colors)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} Comparison')
    ax.set_xlim(0, 1.05)
    
    # 添加数值标签
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{val:.3f}', va='center', fontsize=9)

# 训练时间对比
ax = axes[1, 2]
sorted_time = results_df.sort_values('Fit_Time', ascending=True)
bar_colors = ['#2196F3' if 'TabPFN' in m else '#90A4AE' for m in sorted_time['Model']]
bars = ax.barh(sorted_time['Model'], sorted_time['Fit_Time'], color=bar_colors)
ax.set_xlabel('Training Time (seconds)')
ax.set_title('Training Time Comparison')
for bar, val in zip(bars, sorted_time['Fit_Time']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}s', va='center', fontsize=9)

plt.suptitle('Pregenant Dataset - Model Benchmark Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('pregenant_benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_benchmark_comparison.png")

In [ ]:
# ROC 曲线对比
print("\n生成 ROC 曲线...")

fig, ax = plt.subplots(figsize=(10, 8))

for name in results_df['Model']:
    if name in probabilities:
        fpr, tpr, _ = roc_curve(y_test, probabilities[name])
        auc_score = roc_auc_score(y_test, probabilities[name])
        color = '#2196F3' if 'TabPFN' in name else None
        linewidth = 3 if 'TabPFN' in name else 1.5
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc_score:.3f})", 
                color=color, linewidth=linewidth)

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves Comparison')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.savefig('pregenant_roc_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_roc_curves.png")

## 3. SHAP 解释性分析

In [ ]:
# 使用 TabPFN 进行 SHAP 分析 (使用全部数据)
print("训练 TabPFN 模型用于 SHAP 分析...")

# 重新训练 TabPFN
tabpfn_model = TabPFNClassifier.create_default_for_version(
    ModelVersion.V2, n_estimators=4, device='cpu'
)
tabpfn_model.fit(X_train, y_train)

# 使用 KernelExplainer 进行 SHAP 分析
print("计算 SHAP 值 (使用全部 40 个样本，这可能需要一些时间)...")

# 使用训练集作为背景样本
background = X_train.iloc[:min(20, len(X_train))]

# 使用全部数据计算 SHAP 值，而非仅测试集子集
shap_sample = X.copy()

explainer = shap.KernelExplainer(tabpfn_model.predict_proba, background)
shap_values = explainer.shap_values(shap_sample, nsamples=100)

# 处理 SHAP 值维度 (关键修复)
shap_values_array = np.array(shap_values)
print(f"SHAP values shape: {shap_values_array.shape}")

# 提取类别 1 (正类) 的 SHAP 值
if shap_values_array.ndim == 3:
    # 形状: (n_samples, n_features, n_classes)
    shap_values_class1 = shap_values_array[:, :, 1]
elif isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

print(f"Class 1 SHAP values shape: {shap_values_class1.shape}")
print(f"✓ SHAP 计算完成! 共 {len(shap_sample)} 个样本")

In [ ]:
# SHAP Summary Plot (Bar) - 最稳定的图形
print("\n生成 SHAP Bar Plot...")

fig, ax = plt.subplots(figsize=(10, 12))

# 使用 shap.summary_plot (bar模式)
shap.summary_plot(
    shap_values_class1,
    shap_sample,
    feature_names=feature_names,
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title('Top 20 Important Metabolites (Mean |SHAP|)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pregenant_shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_shap_bar.png")

In [ ]:
# SHAP Beeswarm Plot - 使用 matplotlib 手动绘制 (全部 40 个样本)
print("\n生成 SHAP Beeswarm Plot...")

# 计算每个特征的平均绝对 SHAP 值
mean_shap = np.abs(shap_values_class1).mean(axis=0)

# 获取 Top 20 特征索引
top_indices = np.argsort(mean_shap)[-20:][::-1]

# 提取 Top 20 的 SHAP 值和特征名称
top_shap = shap_values_class1[:, top_indices]
top_features = [feature_names[i] for i in top_indices]
top_data = shap_sample.iloc[:, top_indices].values

# 创建图形
fig, ax = plt.subplots(figsize=(10, 10))

# 手动绘制 beeswarm-like plot
y_positions = np.arange(len(top_features))

for i, (feature_idx, y_pos) in enumerate(zip(top_indices, y_positions)):
    shap_vals = shap_values_class1[:, feature_idx]
    feature_vals = shap_sample.iloc[:, feature_idx].values
    
    # 根据特征值着色
    val_range = feature_vals.max() - feature_vals.min()
    if val_range > 0:
        normed = (feature_vals - feature_vals.min()) / val_range
    else:
        normed = np.full_like(feature_vals, 0.5)
    colors = plt.cm.RdBu_r(normed)
    
    # 添加一些随机抖动避免重叠
    jitter = np.random.normal(0, 0.05, len(shap_vals))
    
    ax.scatter(shap_vals, [y_pos] * len(shap_vals) + jitter, c=colors, alpha=0.7, s=40, edgecolors='black', linewidth=0.3)

ax.set_yticks(y_positions)
ax.set_yticklabels(top_features, fontsize=10)
ax.set_xlabel('SHAP Value (Impact on Prediction)', fontsize=12)
ax.set_title(f'SHAP Beeswarm - Top 20 Metabolites (n={len(shap_sample)})', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='x')

# 添加颜色条说明
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20)
cbar.set_label('Feature Value (Low → High)', rotation=270, labelpad=15)

plt.tight_layout()
plt.savefig('pregenant_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"✓ 已保存: preganent_shap_beeswarm.png (共 {len(shap_sample)} 个样本点)")

In [ ]:
# 获取最重要的特征
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': mean_shap
}).sort_values('importance', ascending=False)

print("\n" + "="*60)
print("TOP 20 重要代谢物 (基于 SHAP)")
print("="*60)
print(feature_importance.head(20).to_string(index=False))

# 保存结果
feature_importance.to_csv('pregenant_feature_importance.csv', index=False)
print("\n✓ 特征重要性已保存到: preganent_feature_importance.csv")

In [ ]:
# Dependence Plot - Top 2 特征 (使用全部数据)
print("\n生成 SHAP Dependence Plots...")

top_features = feature_importance.head(2)['feature'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, feature in enumerate(top_features):
    feature_idx = feature_names.index(feature)
    
    shap_vals = shap_values_class1[:, feature_idx]
    feature_vals = shap_sample.iloc[:, feature_idx].values
    
    # 散点图
    scatter = axes[idx].scatter(
        feature_vals, shap_vals, 
        c=feature_vals, cmap='RdBu_r', 
        alpha=0.7, s=80, edgecolors='black', linewidth=0.5
    )
    
    axes[idx].set_xlabel(f'{feature} Value', fontsize=11)
    axes[idx].set_ylabel('SHAP Value', fontsize=11)
    axes[idx].set_title(f'SHAP Dependence: {feature}', fontsize=12, fontweight='bold')
    axes[idx].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[idx].grid(True, alpha=0.3)
    
    # 添加颜色条
    cbar = plt.colorbar(scatter, ax=axes[idx], shrink=0.6)
    cbar.set_label('Feature Value', rotation=270, labelpad=15)

plt.tight_layout()
plt.savefig('pregenant_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_shap_dependence.png")

## 4. 代谢物分布分析

In [ ]:
# Top 重要代谢物在两类样本中的分布
print("\n生成代谢物分布图...")

top_10_features = feature_importance.head(10)['feature'].tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, feature in enumerate(top_10_features):
    ax = axes[idx]
    
    # 获取原始值 (非对数)
    feature_data = df[[feature, 'Pregent']].copy()
    feature_data[feature] = pd.to_numeric(feature_data[feature], errors='coerce')
    feature_data = feature_data.dropna()
    
    # 箱线图
    box_data = [
        feature_data[feature_data['Pregent'] == 'N'][feature].values,
        feature_data[feature_data['Pregent'] == 'Y'][feature].values
    ]
    bp = ax.boxplot(box_data, labels=['N', 'Y'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightcoral')
    bp['boxes'][1].set_facecolor('lightblue')
    
    ax.set_title(f'{feature}', fontsize=10)
    ax.set_xlabel('Pregnant')
    ax.set_ylabel('Concentration')

plt.suptitle('Top 10 Metabolites Distribution by Group', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pregenant_metabolite_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_metabolite_distribution.png")

In [ ]:
# 热力图 - Top 20 代谢物相关性
print("\n生成相关性热力图...")

top_20_features = feature_importance.head(20)['feature'].tolist()
top_indices = [feature_names.index(f) for f in top_20_features]

# 计算相关性
corr_matrix = X.iloc[:, top_indices].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

# 设置刻度
ax.set_xticks(np.arange(len(top_20_features)))
ax.set_yticks(np.arange(len(top_20_features)))
ax.set_xticklabels(top_20_features, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(top_20_features, fontsize=8)

# 添加颜色条
cbar = plt.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label('Correlation', rotation=270, labelpad=15)

# 添加数值标注
for i in range(len(top_20_features)):
    for j in range(len(top_20_features)):
        text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                      ha="center", va="center", color="black", fontsize=6)

ax.set_title('Correlation Matrix - Top 20 Important Metabolites', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pregenant_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_correlation_heatmap.png")

## 5. 交叉验证评估

In [ ]:
# 5-fold 交叉验证
from sklearn.model_selection import cross_val_score

cv_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("5-Fold Cross Validation Results\n" + "="*60)

for name, model in models.items():
    print(f"\n[{name}] ", end="")
    try:
        # 交叉验证准确率
        cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
        cv_results.append({
            'Model': name,
            'CV_Mean': cv_scores.mean(),
            'CV_Std': cv_scores.std(),
            'CV_Scores': cv_scores
        })
        print(f"Mean Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        continue

cv_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'CV_Scores'} for r in cv_results])
cv_df = cv_df.sort_values('CV_Mean', ascending=False)

In [ ]:
# 交叉验证结果可视化
print("\n生成交叉验证结果图...")

fig, ax = plt.subplots(figsize=(12, 6))

models_sorted = [r['Model'] for r in sorted(cv_results, key=lambda x: x['CV_Mean'])]
means = [r['CV_Mean'] for r in sorted(cv_results, key=lambda x: x['CV_Mean'])]
stds = [r['CV_Std'] for r in sorted(cv_results, key=lambda x: x['CV_Mean'])]
colors = ['#2196F3' if 'TabPFN' in m else '#90A4AE' for m in models_sorted]

bars = ax.barh(models_sorted, means, xerr=stds, capsize=5, color=colors, alpha=0.8)
ax.set_xlabel('CV Accuracy')
ax.set_title('5-Fold Cross Validation Accuracy (Mean ± Std)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)

for bar, mean, std in zip(bars, means, stds):
    ax.text(mean + std + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{mean:.3f}±{std:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('pregenant_cv_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 已保存: preganent_cv_results.png")

## 6. 结果汇总

In [ ]:
# 综合结果汇总
print("\n" + "="*80)
print("PREGENANT 代谢组学数据分析 - 结果汇总")
print("="*80)

print("\n📊 数据集信息:")
print(f"  - 总样本数: {len(df)}")
print(f"  - 特征数: {len(feature_names)} (代谢物)")
print(f"  - 正例 (Y): {sum(y==1)} ({sum(y==1)/len(y)*100:.1f}%)")
print(f"  - 负例 (N): {sum(y==0)} ({sum(y==0)/len(y)*100:.1f}%)")

print("\n🏆 Benchmark 排名 (按 Accuracy):")
for i, (_, row) in enumerate(results_df.head(5).iterrows(), 1):
    marker = "⭐" if 'TabPFN' in row['Model'] else "  "
    print(f"  {marker} {i}. {row['Model']}: {row['Accuracy']:.4f}")

print("\n🔬 Top 5 重要代谢物 (SHAP):")
for i, (_, row) in enumerate(feature_importance.head(5).iterrows(), 1):
    print(f"  {i}. {row['feature']}: {row['importance']:.4f}")

print("\n📈 交叉验证性能:")
for i, (_, row) in enumerate(cv_df.head(5).iterrows(), 1):
    marker = "⭐" if 'TabPFN' in row['Model'] else "  "
    print(f"  {marker} {i}. {row['Model']}: {row['CV_Mean']:.4f} ± {row['CV_Std']:.4f}")

print("\n✅ 分析完成!")
print("="*80)

In [ ]:
# 保存所有结果
results_df.to_csv('pregenant_benchmark_results.csv', index=False)
cv_df.to_csv('pregenant_cv_results.csv', index=False)

print("\n📁 输出文件:")
print("  CSV 文件:")
print("    - preganent_benchmark_results.csv")
print("    - preganent_feature_importance.csv")
print("    - preganent_cv_results.csv")
print("\n  可视化图表:")
print("    - preganent_benchmark_comparison.png")
print("    - preganent_roc_curves.png")
print("    - preganent_shap_beeswarm.png")
print("    - preganent_shap_bar.png")
print("    - preganent_shap_dependence.png")
print("    - preganent_metabolite_distribution.png")
print("    - preganent_correlation_heatmap.png")
print("    - preganent_cv_results.png")